# CMT — large300 — Notebook corrigé & développé

**Nouveaux objectifs** (notes manuscrites) :

1. **Zoom 0–15 km** propre sur $\bar u$, $\partial_z\bar u$, $\partial_z^2\bar u$, $\bar w$, flux de Reynolds.
2. **Correction de la POD** — *on doit retrouver le flux de Reynolds*. Le bug de l'ancienne version : la POD était faite sur $\psi$ **moyennée en $y$**, ce qui détruit la covariance $u'w'$ ; on ne peut donc pas reconstruire le flux. **Correctif** : POD conjointe des **anomalies** $[u';\,w']$ (résolues en $x$), d'où une reconstruction fidèle du flux.
3. **$\psi'$ vue comme un relief** — chercher une meilleure base de décomposition (DCT 2D / Fourier) pour ce type de champ, décomposer le relief $\psi'$, puis reconstruire le flux de Reynolds ; comparer à la POD.
4. **Zones de fort cisaillement (0–15 km)** — critère $|\partial_z\bar u|>$ seuil (moyenné en $y$) : où sont les $x$ et $t$ ? régions continues ? moyennes régionales, puis **fermeture linéaire locale**
$$\partial_z\!\big(\rho_0\,\overline{u'w'}\big) = \alpha(z,t)\,\bar u(z,t) + \beta(z,t)\,\partial_z\bar u(z,t) + \gamma(z,t)\,\partial_z^2\bar u,$$
testée par moindres carrés (globale, par altitude, par instant), avec validation train/test.

Conventions inchangées : lecture niveau par niveau, $\rho_0$ via $T_v$, masques PRW, **imports limités à numpy / scipy / xarray / matplotlib / gc / os**, blocs auto-suffisants, sorties toujours affichées.


## 0. Imports & configuration

Pipeline **large300** habituel. Seul ajout d'import : `scipy.fft` (DCT 2D) — reste dans scipy, pas de nouvelle dépendance.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import xarray as xr
import gc, os
from scipy.optimize import curve_fit
from scipy.sparse import diags, kron, identity
from scipy.sparse.linalg import spsolve
from scipy.ndimage import label as ndi_label
from scipy.fft import dctn, idctn          # <-- pour la base "relief" (obj. 3)

plt.rcParams.update({
    'figure.dpi': 110, 'font.size': 11,
    'axes.grid': True, 'grid.alpha': 0.25,
    'image.cmap': 'RdBu_r',
})

# ============================================================
#  CONFIGURATION  (identique au notebook rcemip_large300)
# ============================================================
DIR_3D = '3D';  DIR_2D = '2D';  DIR_1D = '1D'
def path3d(var): return os.path.join(DIR_3D, f'MESONH_RCE_large300_3D_{var}.nc')
def path2d(var): return os.path.join(DIR_2D, f'MESONH_RCE_large300_2D_{var}.nc')
def path1d(var): return os.path.join(DIR_1D, f'MESONH_RCE_large300_1D_{var}.nc')

Rd, Rv = 287.05, 461.5
EPSILON = Rd / Rv          # ~ 0.622
BLOC    = 2                # taille de bloc temporel (RAM)

Z_TOP_KM = 15.0            # <-- fenetre d'analyse "0-15 km" des nouvelles consignes

print('Config prete.  (fenetre principale 0-%.0f km)' % Z_TOP_KM)

In [ ]:
# --- (0a) Métadonnées : dims, tailles, altitude, fenêtre stationnaire ---
_ds = xr.open_dataset(path3d('ua'));  _da = _ds['ua']
dim_t, dim_z, dim_y, dim_x = _da.dims        # ordre (t, z, y, x)
n_t = _da.sizes[dim_t];  n_z = _da.sizes[dim_z]
n_y = _da.sizes[dim_y];  n_x = _da.sizes[dim_x]
_ds.close();  del _ds, _da;  gc.collect()

_t1 = xr.open_dataset(path1d('ua_avg'))
alt = _t1['altitude'].values.astype(float).copy()
_t1.close()

t_stat   = int(2 * n_t / 3)          # dernier tiers = stationnaire
idx_stat = slice(t_stat, None)
n_stat   = n_t - t_stat
print(f'Grille : {n_t} t x {n_z} z x {n_y} y x {n_x} x')
print(f'Altitude : {alt[0]:.0f} -> {alt[-1]:.0f} m')
print(f'Stationnaire : t={t_stat}->{n_t-1} ({n_stat} pas)')

# grille horizontale (m) — dx/dy = 1000 m par défaut si coords = indices
try:
    xcoord = _ds.coords[dim_x].values.astype(float)
    dx = float(xcoord[1] - xcoord[0])
    if dx < 10:  # coords = indices entiers -> fallback
        raise ValueError
except Exception:
    dx = 1000.0
dy = dx
x = np.arange(n_x) * dx
print(f'dx = dy = {dx:.0f} m')

In [ ]:
# --- (0b) Profil rho_0(z) via température virtuelle (gaz parfaits) ---
rho0 = np.zeros(n_z)
ds_ta  = xr.open_dataset(path3d('ta'))
ds_pa  = xr.open_dataset(path3d('pa'))
ds_hus = xr.open_dataset(path3d('hus'))
for iz in range(n_z):
    ta_z  = ds_ta['ta'].isel({dim_z: iz, dim_t: idx_stat}).values
    pa_z  = ds_pa['pa'].isel({dim_z: iz, dim_t: idx_stat}).values
    hus_z = ds_hus['hus'].isel({dim_z: iz, dim_t: idx_stat}).values
    tv = ta_z * (1.0 + (1.0 / EPSILON - 1.0) * hus_z)
    rho0[iz] = np.mean(pa_z / (Rd * tv))
ds_ta.close(); ds_pa.close(); ds_hus.close(); gc.collect()

fig, ax = plt.subplots(figsize=(4, 5))
ax.plot(rho0, alt / 1000)
ax.set_xlabel(r'$\rho_0$ (kg/m$^3$)'); ax.set_ylabel('z (km)')
ax.set_title(r'Profil $\rho_0(z)$')
fig.tight_layout(); plt.show()

In [ ]:
# --- (0c) Masques humide / sec (PRW), 2D (y,x), seuil = médiane ---
ds_prw = xr.open_dataset(path2d('prw'))
prw_t  = ds_prw['prw'].isel({dim_t: idx_stat}).values     # (n_stat, ny, nx)
ds_prw.close(); gc.collect()
prw = prw_t.mean(axis=0)                                   # moyenne temporelle (ny,nx)
seuil_prw = np.median(prw)
mh = prw > seuil_prw     # humide
ms = ~mh                 # sec
print(f'Seuil PRW = {seuil_prw:.1f} kg/m2  -  {mh.mean()*100:.0f}% de colonnes humides')

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.pcolormesh(prw, cmap='YlGnBu', shading='auto')
ax.contour(mh.astype(float), levels=[0.5], colors='k', linewidths=1)
ax.set_title('PRW moyen + contour humide/sec')
fig.colorbar(im, ax=ax, label='PRW (kg/m2)')
fig.tight_layout(); plt.show()

## 1. Visualisation — zoom 0–15 km

Profils $\rho_0\langle u'w'\rangle$, $\bar u$, $\partial_z\bar u$, $\partial_z^2\bar u$, $\bar w$ à un instant et en superposition temporelle, **cadrés sur 0–15 km**. Dérivées calculées sur la colonne complète puis tronquées (pas d'artefact de bord).


In [ ]:
# --- Profils rho0<u'w'>(z,t), ubar(z,t), wbar(z,t), niveau par niveau ---
flux  = np.zeros((n_z, n_stat))
ubar  = np.zeros((n_z, n_stat))
wbar  = np.zeros((n_z, n_stat))

ds_u = xr.open_dataset(path3d('ua'))
ds_w = xr.open_dataset(path3d('wa'))
for iz in range(n_z):
    u = ds_u['ua'].isel({dim_z: iz, dim_t: idx_stat}).values   # (n_stat, ny, nx)
    w = ds_w['wa'].isel({dim_z: iz, dim_t: idx_stat}).values
    ub = u.mean(axis=(1, 2))
    wb = w.mean(axis=(1, 2))
    up = u - ub[:, None, None]
    wp = w - wb[:, None, None]
    flux[iz, :] = rho0[iz] * (up * wp).mean(axis=(1, 2))
    ubar[iz, :] = ub
    wbar[iz, :] = wb
ds_u.close(); ds_w.close(); gc.collect()

dudz   = np.gradient(ubar, alt, axis=0)
d2udz2 = np.gradient(dudz, alt, axis=0)
print('flux, ubar, wbar, dudz, d2udz2 prêts :', flux.shape)

In [ ]:
# --- masque d'affichage 0-15 km (les champs restent calcules sur toute la colonne) ---
zkm    = alt / 1000.0
mask_show = zkm <= Z_TOP_KM
alt_s  = alt[mask_show]

def dashboard_at_time(it, zmax=Z_TOP_KM):
    """Profils verticaux a l'instant it, zoom 0-zmax km."""
    fig, axes = plt.subplots(1, 5, figsize=(17, 5), sharey=True)
    panels = [
        (flux,   r"$\rho_0\langle u'w'\rangle$"),
        (dudz,   r"$\partial_z \bar u$"),
        (ubar,   r"$\bar u$ (m/s)"),
        (wbar,   r"$\bar w$ (m/s)"),
        (d2udz2, r"$\partial_z^2 \bar u$"),
    ]
    for ax, (field, label) in zip(axes, panels):
        ax.plot(field[mask_show, it], zkm[mask_show], lw=1.6)
        ax.axvline(0, color='k', lw=0.5)
        ax.set_xlabel(label)
    axes[0].set_ylabel('z (km)')
    axes[0].set_ylim(0, zmax)
    fig.suptitle(f'Diagnostics 0-{zmax:.0f} km  -  instant {it} (t={t_stat+it})')
    fig.tight_layout(); plt.show()

dashboard_at_time(n_stat // 2)

In [ ]:
def dashboard_multi_time(its, zmax=Z_TOP_KM, cmap_name='viridis'):
    """Superposition de plusieurs instants (couleur = temps), zoom 0-zmax km."""
    cmap = plt.get_cmap(cmap_name)
    colors = cmap(np.linspace(0, 1, len(its)))
    fig, axes = plt.subplots(1, 5, figsize=(17, 5), sharey=True)
    panels = [
        (flux,   r"$\rho_0\langle u'w'\rangle$"),
        (dudz,   r"$\partial_z \bar u$"),
        (ubar,   r"$\bar u$ (m/s)"),
        (wbar,   r"$\bar w$ (m/s)"),
        (d2udz2, r"$\partial_z^2 \bar u$"),
    ]
    for ax, (field, label) in zip(axes, panels):
        for it, c in zip(its, colors):
            ax.plot(field[mask_show, it], zkm[mask_show], lw=1.2, color=c)
        ax.axvline(0, color='k', lw=0.5)
        ax.set_xlabel(label)
    axes[0].set_ylabel('z (km)'); axes[0].set_ylim(0, zmax)
    sm = plt.cm.ScalarMappable(cmap=cmap,
            norm=plt.Normalize(vmin=t_stat, vmax=t_stat + n_stat - 1))
    fig.colorbar(sm, ax=axes, label='indice temporel global', pad=0.01, aspect=30)
    fig.suptitle(f'Diagnostics 0-{zmax:.0f} km  -  {len(its)} instants')
    plt.show()

its = np.linspace(0, n_stat - 1, 10, dtype=int)
dashboard_multi_time(its)

In [ ]:
# --- Hovmoller z-t (0-15 km) : structure temps-altitude du flux et du cisaillement ---
tt = np.arange(n_stat) + t_stat
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, (field, label) in zip(axes, [
        (flux,   r"$\rho_0\langle u'w'\rangle$"),
        (dudz,   r"$\partial_z\bar u$"),
        (ubar,   r"$\bar u$")]):
    F = field[mask_show]
    vmax = np.nanpercentile(np.abs(F), 98) + 1e-30
    im = ax.pcolormesh(tt, zkm[mask_show], F, cmap='RdBu_r',
                       norm=TwoSlopeNorm(0, -vmax, vmax), shading='auto')
    ax.set_title(label); ax.set_xlabel('t (indice global)')
    fig.colorbar(im, ax=ax, pad=0.02)
axes[0].set_ylabel('z (km)'); axes[0].set_ylim(0, Z_TOP_KM)
fig.suptitle('Hovmoller z-t (0-15 km)')
fig.tight_layout(); plt.show()

## 2. Equation discovery (STLSQ) — termes grande échelle

Inchangé (méthode déjà validée) : bibliothèque de candidats *grande échelle*, STLSQ codé à la main, restriction 2–18 km pour la découverte d'équation, puis poids relatif de chaque terme retenu.


In [ ]:
# ============================================================
# OBJECTIF 2 — Equation discovery, restreint à 2-18 km
# ============================================================

# --- restriction verticale, appliquée aux grandeurs déjà calculées sur la
#     grille complète (dudz/d2udz2 restent dérivés sur toute la colonne,
#     puis on tronque — ça évite les artefacts de bord d'une dérivation
#     faite directement sur la fenêtre réduite) ---
mask_z = (alt >= 2000) & (alt <= 18000)
alt_r, rho0_r, ubar_r, wbar_r, dudz_r, d2udz2_r, flux_r = (
    a[mask_z] for a in (alt, rho0, ubar, wbar, dudz, d2udz2, flux)
)
n_z_r = alt_r.size
print(f'{n_z_r} niveaux entre {alt_r[0]:.0f} et {alt_r[-1]:.0f} m (sur {n_z} au total)')

d3udz3_r  = np.gradient(d2udz2, alt, axis=0)[mask_z]
dwdz_r    = np.gradient(wbar, alt, axis=0)[mask_z]
drho0dz_r = np.gradient(rho0, alt)[mask_z][:, None]


def build_library():
    '''Bibliothèque grande échelle, restreinte à 2-18 km.'''
    Z = np.tile(alt_r[:, None], (1, n_stat))
    H = alt.max()                          # sommet convectif = pleine colonne (référence physique)
    rho0_2d = np.tile(rho0_r[:, None], (1, n_stat))

    terms = {
        'dudz':         dudz_r,
        'd2udz2':       d2udz2_r,
        'd3udz3':       d3udz3_r,
        'dudz2':        dudz_r ** 2,
        'absdudz_dudz': np.abs(dudz_r) * dudz_r,
        'dudz_d2udz2':  dudz_r * d2udz2_r,
        'd2udz2_2':     d2udz2_r ** 2,
        'u':        ubar_r,
        'u2':       ubar_r ** 2,
        'u3':       ubar_r ** 3,
        'u_dudz':   ubar_r * dudz_r,
        'u_d2udz2': ubar_r * d2udz2_r,
        'u2_dudz':  ubar_r ** 2 * dudz_r,
        'w':        wbar_r,
        'w2':       wbar_r ** 2,
        'w_dudz':   wbar_r * dudz_r,
        'w_d2udz2': wbar_r * d2udz2_r,
        'dwdz':     dwdz_r,
        'w_u':      wbar_r * ubar_r,
        'rho0_dudz':    rho0_2d * dudz_r,
        'rho0_d2udz2':  rho0_2d * d2udz2_r,
        'drho0dz_u':    drho0dz_r * ubar_r,
        'drho0dz_dudz': drho0dz_r * dudz_r,
        'z_dudz':   Z * dudz_r,
        'z2_dudz':  Z ** 2 * dudz_r,
        'z_d2udz2': Z * d2udz2_r,
        'zHz_dudz': Z * (H - Z) * dudz_r,
        'zHz':      (Z / H) * (1 - Z / H),
    }
    names = list(terms.keys())
    Theta = np.stack([terms[n].ravel() for n in names], axis=1)
    return Theta, names


def nse(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return 1.0 - ss_res / ss_tot


def stlsq(Theta, y, threshold=0.05, n_iter=15, alpha=1e-8):
    n_terms = Theta.shape[1]
    coef = np.linalg.solve(Theta.T @ Theta + alpha * np.eye(n_terms), Theta.T @ y)
    active = np.ones(n_terms, dtype=bool)
    for _ in range(n_iter):
        small = np.abs(coef) < threshold
        if not np.any(small & active):
            break
        active[small] = False
        if not np.any(active):
            break
        Xs = Theta[:, active]
        coef_active = np.linalg.solve(Xs.T @ Xs + alpha * np.eye(Xs.shape[1]), Xs.T @ y)
        coef = np.zeros(n_terms)
        coef[active] = coef_active
    return coef, active


# --- fit ---
Theta, names = build_library()
y = flux_r.ravel()

mu, sigma = Theta.mean(axis=0), Theta.std(axis=0) + 1e-12
Theta_n = (Theta - mu) / sigma
y_n = (y - y.mean()) / (y.std() + 1e-12)

coef_n, active = stlsq(Theta_n, y_n, threshold=0.08)
coef = coef_n * (y.std() / sigma)
intercept = y.mean() - mu @ coef

print('Termes retenus :')
for name, c, a in zip(names, coef, active):
    if a:
        print(f'  {name:15s} : {c:+.4e}')
print(f'  intercept       : {intercept:+.4e}')

flux_pred = (Theta @ coef + intercept).reshape(n_z_r, n_stat)
r2 = nse(flux_r, flux_pred)
print(f'\nNSE (2-18 km, z-t complet) = {r2:.3f}')


fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(flux_r.mean(axis=1), alt_r / 1000, label='flux vrai (moy. temp.)')
ax.plot(flux_pred.mean(axis=1), alt_r / 1000, '--', label='régression (moy. temp.)')
ax.axvline(0, color='k', lw=0.5)
ax.set_xlabel(r"$\rho_0\langle u'w'\rangle$"); ax.set_ylabel('z (km)')
ax.legend(); ax.set_title('Profil moyen 2-18 km : vrai vs régression')
fig.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(flux_r.ravel(), flux_pred.ravel(), s=4, alpha=0.3)
lims = [min(flux_r.min(), flux_pred.min()), max(flux_r.max(), flux_pred.max())]
ax.plot(lims, lims, 'k--', lw=1)
ax.set_xlabel('flux vrai'); ax.set_ylabel('flux régression')
ax.set_title(f'NSE = {r2:.3f}')
fig.tight_layout(); plt.show()

In [ ]:
# ============================================================
# Contribution en pourcentage de chaque terme retenu
# ============================================================

importance = {}
for j, (name, c, a) in enumerate(zip(names, coef, active)):
    if not a:
        continue
    contrib = (Theta[:, j] * c).reshape(n_z_r, n_stat).mean(axis=1)   # profil moyen dans le temps
    importance[name] = np.sum(np.abs(contrib))                        # poids intégré sur 2-18 km

importance['intercept'] = np.sum(np.abs(np.full(n_z_r, intercept)))

total = sum(importance.values())
pct = {name: 100 * v / total for name, v in importance.items()}
pct_sorted = dict(sorted(pct.items(), key=lambda kv: kv[1], reverse=True))

print('Contribution relative (%) au profil moyen 2-18 km :')
for name, p in pct_sorted.items():
    print(f'  {name:15s} : {p:5.1f}%')

fig, ax = plt.subplots(figsize=(6, 0.4 * len(pct_sorted) + 1))
bars = ax.barh(list(pct_sorted.keys())[::-1], list(pct_sorted.values())[::-1], color='steelblue')
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
ax.set_xlabel('contribution relative au profil moyen (%)')
ax.set_title('Poids de chaque terme (2-18 km)')
ax.set_xlim(0, max(pct_sorted.values()) * 1.15)
fig.tight_layout(); plt.show()

## 3. POD corrigée — reconstruction du flux de Reynolds

**Diagnostic du bug.** L'ancienne POD portait sur $\psi(x,z,t)$ **moyenné en $y$**. Or
$$\rho_0\,\overline{u'w'} = \rho_0\,\overline{(u-\bar u)(w-\bar w)}$$
est une **covariance des fluctuations** : moyenner en $y$ *avant* la POD détruit la corrélation qui produit le flux. On ne peut donc pas le reconstruire — c'est le « problème de calcul » des notes.

**Correctif.** POD **conjointe des champs d'anomalie** $[\,u'(x,z);\ w'(x,z)\,]$ (résolus en $x$). Méthode des snapshots (centrée). La reconstruction tronquée à $N$ modes redonne alors $\rho_0\langle u'w'\rangle$, ce qu'on vérifie par le NSE.


In [ ]:
# --- construction des snapshots d'ANOMALIE u'(x,z,t), w'(x,z,t) (moyenne-y) ---
# fluctuation = champ - moyenne horizontale <.>_x  (a chaque z, chaque t)
STRIDE_T = max(1, n_stat // 60)                 # ~60 instants au plus
t_idx_pod = np.arange(0, n_stat, STRIDE_T)
Np = len(t_idx_pod)

up_snaps = np.zeros((Np, n_z, n_x))
wp_snaps = np.zeros((Np, n_z, n_x))
ds_u = xr.open_dataset(path3d('ua'))
ds_w = xr.open_dataset(path3d('wa'))
for i_snap, it in enumerate(t_idx_pod):
    itg = t_stat + it
    U2d = ds_u['ua'].isel({dim_t: itg}).mean(dim=dim_y).values      # (nz, nx)
    W2d = ds_w['wa'].isel({dim_t: itg}).mean(dim=dim_y).values
    up_snaps[i_snap] = U2d - U2d.mean(axis=1, keepdims=True)        # anomalie en x
    wp_snaps[i_snap] = W2d - W2d.mean(axis=1, keepdims=True)
ds_u.close(); ds_w.close(); gc.collect()
print('snapshots anomalie :', up_snaps.shape)

# flux "vrai" 3D deja calcule (obj.1), meme sous-echantillon temporel, pour la reference
flux_true_profile = flux[:, t_idx_pod].mean(axis=1)                 # (nz,)

In [ ]:
# --- POD conjointe [u';w'] par methode des snapshots (centree) ---
def nse(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return 1.0 - ss_res / ss_tot

half = n_z * n_x
S = np.concatenate([up_snaps.reshape(Np, -1),
                    wp_snaps.reshape(Np, -1)], axis=1)               # (Np, 2*nz*nx)
Smean = S.mean(axis=0)
A = S - Smean

Cs = A @ A.T / Np
evals, evecs = np.linalg.eigh(Cs)
order = evals.argsort()[::-1]
evals = np.clip(evals[order], 0, None); evecs = evecs[:, order]

modes_flat = evecs.T @ A
norms = np.linalg.norm(modes_flat, axis=1) + 1e-30
modes_flat = modes_flat / norms[:, None]
a_n = A @ modes_flat.T                                              # (Np, Np) coeffs temporels

energy = evals / evals.sum(); cum_energy = np.cumsum(energy)
n_modes = int(np.searchsorted(cum_energy, 0.90) + 1)
print(f'{n_modes} modes pour 90% de l energie (sur {len(cum_energy)}).')

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(np.arange(1, len(cum_energy) + 1), cum_energy, 'o-', ms=3)
ax.axhline(0.90, color='grey', ls='--', lw=0.8)
ax.axvline(n_modes, color='grey', ls='--', lw=0.8)
ax.set_xlabel('nombre de modes'); ax.set_ylabel('energie cumulee')
ax.set_title("Troncature POD (anomalies u',w')")
fig.tight_layout(); plt.show()

In [ ]:
# --- reconstruction du flux a partir des N premiers modes ---
def reconstruct_flux_pod(N):
    """Reco de u',w' avec N modes -> profil rho0<u'w'>(z)."""
    rec = Smean[None, :] + a_n[:, :N] @ modes_flat[:N]              # (Np, 2*nz*nx)
    up_r = rec[:, :half].reshape(Np, n_z, n_x)
    wp_r = rec[:, half:].reshape(Np, n_z, n_x)
    up_r = up_r - up_r.mean(axis=2, keepdims=True)                  # coherence def. fluctuation
    wp_r = wp_r - wp_r.mean(axis=2, keepdims=True)
    return (rho0[None, :] * (up_r * wp_r).mean(axis=2)).mean(axis=0)  # (nz,)

# balayage du nombre de modes : NSE(flux) vs N
Ns = np.unique(np.clip(
        np.array([n_modes, 2*n_modes, 4*n_modes, 8*n_modes,
                  Np//2, Np-1]), 1, Np-1))
nse_vs_N = [nse(flux_true_profile, reconstruct_flux_pod(N)) for N in Ns]
for N, s in zip(Ns, nse_vs_N):
    print(f'  N={N:3d} modes  ->  NSE flux = {s:+.3f}')

flux_pod = reconstruct_flux_pod(n_modes)
nse_pod  = nse(flux_true_profile, flux_pod)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].plot(flux_true_profile[mask_show], zkm[mask_show], label='flux complet (3D)')
axes[0].plot(flux_pod[mask_show], zkm[mask_show], '--',
             label=f'{n_modes} modes POD (NSE={nse_pod:.3f})')
axes[0].axvline(0, color='k', lw=0.5); axes[0].set_ylim(0, Z_TOP_KM)
axes[0].set_xlabel(r"$\rho_0\langle u'w'\rangle$"); axes[0].set_ylabel('z (km)')
axes[0].legend(); axes[0].set_title('Reconstruction POD corrigee du flux')

axes[1].plot(Ns, nse_vs_N, 'o-')
axes[1].axhline(0, color='k', lw=0.5)
axes[1].set_xlabel('nombre de modes N'); axes[1].set_ylabel('NSE flux')
axes[1].set_title('Convergence de la reco du flux')
fig.tight_layout(); plt.show()

In [ ]:
# --- visualisation des premiers modes (composante u' du mode) ---
nshow = min(3, max(1, n_modes))
fig, axes = plt.subplots(1, nshow, figsize=(5*nshow, 4), sharey=True)
if nshow == 1: axes = [axes]
for k, ax in enumerate(axes):
    mode_u = modes_flat[k, :half].reshape(n_z, n_x)
    vmax = np.percentile(np.abs(mode_u), 98) + 1e-30
    im = ax.pcolormesh(x/1000, zkm, mode_u, cmap='RdBu_r',
                       norm=TwoSlopeNorm(0, -vmax, vmax), shading='auto')
    ax.set_ylim(0, Z_TOP_KM); ax.set_xlabel('x (km)')
    ax.set_title(f"mode {k+1} : {energy[k]*100:.1f}% energie")
    fig.colorbar(im, ax=ax, pad=0.02)
axes[0].set_ylabel('z (km)')
fig.suptitle("Modes POD (composante u') — structure spatiale")
fig.tight_layout(); plt.show()

## 4. L'anomalie de $\psi$ vue comme un relief

On traite $\psi'(x,z)$ (anomalie de la fonction de courant, moyenne-$y$) comme une **surface / relief** et on cherche une base plus adaptée que la POD *snapshot-par-snapshot* : la **DCT 2D** (base de cosinus, adaptée aux reliefs bornés). On mesure la **compressibilité**, puis on reconstruit $u',w'$ à partir de $\psi'$ tronqué et on vérifie qu'on **retrouve le flux de Reynolds**.

$$\tilde u' = \frac{1}{\rho_0}\partial_z\tilde\psi',\qquad \tilde w' = -\frac{1}{\rho_0}\partial_x\tilde\psi'.$$


In [ ]:
def psi_poisson_periodic(U2d, W2d, xv, zv, rho0v):
    '''nabla^2 psi = d_x(rho0 w) - d_z(rho0 u), periodique en x, Dirichlet en z.'''
    nz, nx = U2d.shape
    dxl = xv[1] - xv[0]
    rW = rho0v[:, None] * W2d
    rU = rho0v[:, None] * U2d
    dxrW = (np.roll(rW, -1, axis=1) - np.roll(rW, 1, axis=1)) / (2 * dxl)
    dzrU = np.gradient(rU, zv, axis=0)
    omega = dxrW - dzrU

    main_x = -2 * np.ones(nx); off_x = np.ones(nx - 1)
    Lx = diags([main_x, off_x, off_x, [1], [1]], [0, 1, -1, nx - 1, -(nx - 1)]) / dxl ** 2
    dz_mean = np.mean(np.diff(zv))
    main_z = -2 * np.ones(nz); off_z = np.ones(nz - 1)
    Lz = diags([main_z, off_z, off_z], [0, 1, -1]) / dz_mean ** 2
    A = (kron(identity(nz), Lx) + kron(Lz, identity(nx))).tolil()
    b = omega.ravel().copy()

    psi_top = np.cumsum(rho0v * U2d.mean(axis=1)) * dz_mean
    for i in range(nx):
        k0 = i;               A.rows[k0] = [k0]; A.data[k0] = [1.0]; b[k0] = 0.0
        k1 = (nz - 1) * nx + i; A.rows[k1] = [k1]; A.data[k1] = [1.0]; b[k1] = psi_top[-1]
    psi = spsolve(A.tocsr(), b).reshape(nz, nx)
    return psi

In [ ]:
# --- psi(x,z,t) moyenne-y sur les memes snapshots, puis anomalie psi' ---
psi_snaps = np.zeros((Np, n_z, n_x))
ds_u = xr.open_dataset(path3d('ua'))
ds_w = xr.open_dataset(path3d('wa'))
for i_snap, it in enumerate(t_idx_pod):
    itg = t_stat + it
    U2d = ds_u['ua'].isel({dim_t: itg}).mean(dim=dim_y).values
    W2d = ds_w['wa'].isel({dim_t: itg}).mean(dim=dim_y).values
    psi_snaps[i_snap] = psi_poisson_periodic(U2d, W2d, x, alt, rho0)
ds_u.close(); ds_w.close(); gc.collect()

psi_mean = psi_snaps.mean(axis=0)            # relief moyen
psi_ano  = psi_snaps - psi_mean[None]        # relief d'anomalie psi'(x,z,t)
print('psi_ano :', psi_ano.shape)

it0 = Np // 2
fig, ax = plt.subplots(figsize=(7, 4))
vmax = np.percentile(np.abs(psi_ano[it0]), 98) + 1e-30
im = ax.pcolormesh(x/1000, zkm, psi_ano[it0], cmap='terrain',
                   shading='auto', vmin=-vmax, vmax=vmax)
ax.set_ylim(0, Z_TOP_KM); ax.set_xlabel('x (km)'); ax.set_ylabel('z (km)')
ax.set_title(r"Relief $\psi'(x,z)$ a un instant"); fig.colorbar(im, ax=ax)
fig.tight_layout(); plt.show()

In [ ]:
# --- compressibilite du relief par DCT 2D : erreur de reco vs fraction de coeffs ---
def dct_truncate(field2d, frac):
    """Garde les frac (0-1) plus gros coeffs DCT (en module), renvoie la reco."""
    C = dctn(field2d, norm='ortho')
    flat = C.ravel()
    k = max(1, int(frac * flat.size))
    idx = np.argsort(np.abs(flat))[::-1][:k]
    Cm = np.zeros_like(flat); Cm[idx] = flat[idx]
    return idctn(Cm.reshape(C.shape), norm='ortho'), k

fracs = np.array([0.005, 0.01, 0.02, 0.05, 0.1])
err_dct = []
for fr in fracs:
    e = []
    for it in range(Np):
        rec, _ = dct_truncate(psi_ano[it], fr)
        e.append(np.sum((psi_ano[it]-rec)**2) / (np.sum(psi_ano[it]**2)+1e-30))
    err_dct.append(np.mean(e))
err_dct = np.array(err_dct)

# comparaison POD sur psi' : erreur de reco avec K modes (~ meme # ddl)
Apsi = psi_ano.reshape(Np, -1)
Apsi_c = Apsi - Apsi.mean(0)
Cpsi = Apsi_c @ Apsi_c.T / Np
evp, evcp = np.linalg.eigh(Cpsi); op = evp.argsort()[::-1]
evp = np.clip(evp[op], 0, None); evcp = evcp[:, op]
mpsi = evcp.T @ Apsi_c; mpsi /= (np.linalg.norm(mpsi, axis=1)+1e-30)[:, None]
apsi = Apsi_c @ mpsi.T
err_pod = []
Ks = np.maximum(1, (fracs * n_z).astype(int))
for K in Ks:
    recon = Apsi.mean(0)[None] + apsi[:, :K] @ mpsi[:K]
    err_pod.append(np.mean(np.sum((Apsi-recon)**2, axis=1) /
                           (np.sum(Apsi**2, axis=1)+1e-30)))
err_pod = np.array(err_pod)

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.loglog(fracs*100, err_dct, 'o-', label='DCT 2D (relief)')
ax.loglog(fracs*100, err_pod, 's--', label='POD (K modes ~ meme # ddl)')
ax.set_xlabel('fraction de coefficients gardes (%)')
ax.set_ylabel(r"erreur relative de reco de $\psi'$")
ax.set_title(r"Compressibilite du relief $\psi'$ : DCT vs POD")
ax.legend(); ax.grid(True, which='both', alpha=0.3)
fig.tight_layout(); plt.show()
for fr, ed, ep in zip(fracs, err_dct, err_pod):
    print(f'  {fr*100:4.1f}% coeffs  ->  err DCT={ed:.4f}   err POD={ep:.4f}')

In [ ]:
# --- reconstruire le flux de Reynolds depuis psi' (complet et tronque DCT) ---
def flux_from_psi(psi_arr):
    """u' = d_z psi'/rho0 , w' = -d_x psi'/rho0  -> profil rho0<u'w'>(z)."""
    up = np.gradient(psi_arr, alt, axis=1) / rho0[None, :, None]
    wp = -np.gradient(psi_arr, x,   axis=2) / rho0[None, :, None]
    up = up - up.mean(axis=2, keepdims=True)
    wp = wp - wp.mean(axis=2, keepdims=True)
    return (rho0[None, :] * (up * wp).mean(axis=2)).mean(axis=0)

flux_psi_full = flux_from_psi(psi_ano)
nse_psi_full  = nse(flux_true_profile, flux_psi_full)

psi_trunc = np.zeros_like(psi_ano)
for it in range(Np):
    psi_trunc[it], _ = dct_truncate(psi_ano[it], 0.02)
flux_psi_dct = flux_from_psi(psi_trunc)
nse_psi_dct  = nse(flux_true_profile, flux_psi_dct)

print(f'NSE flux via psi complet         = {nse_psi_full:+.3f}')
print(f'NSE flux via psi (2% coeffs DCT) = {nse_psi_dct:+.3f}')
print(f"(rappel) NSE flux via POD u',w'   = {nse_pod:+.3f}")

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot(flux_true_profile[mask_show], zkm[mask_show], 'k', lw=2, label='flux complet 3D')
ax.plot(flux_psi_full[mask_show], zkm[mask_show], '-',  label=f'psi complet ({nse_psi_full:.2f})')
ax.plot(flux_psi_dct[mask_show],  zkm[mask_show], '--', label=f'psi 2% DCT ({nse_psi_dct:.2f})')
ax.plot(flux_pod[mask_show],      zkm[mask_show], ':',  label=f"POD u',w' ({nse_pod:.2f})")
ax.axvline(0, color='k', lw=0.5); ax.set_ylim(0, Z_TOP_KM)
ax.set_xlabel(r"$\rho_0\langle u'w'\rangle$"); ax.set_ylabel('z (km)')
ax.legend(fontsize=9); ax.set_title('Flux reconstruit : 3 voies')
fig.tight_layout(); plt.show()

## 5. Zones de fort cisaillement (0–15 km) & fermeture locale

**Critère** : $|\partial_z\bar u(z,t)| > $ seuil ($\bar u$ moyenné horizontalement). On repère les $(z,t)$ concernés, on regarde si les régions sont **continues en $z$**, on fait des **moyennes régionales**, puis on teste la **fermeture linéaire locale** de la divergence de flux
$$F(z,t)\equiv\partial_z\!\big(\rho_0\overline{u'w'}\big)=\alpha\,\bar u+\beta\,\partial_z\bar u+\gamma\,\partial_z^2\bar u$$
selon trois lectures des coefficients : (a) **constants** (fit global sur les zones fortes), (b) $\alpha,\beta,\gamma(z)$ (un fit par **altitude**, régression sur $t$), (c) $\alpha,\beta,\gamma(t)$ (un fit par **instant**, régression sur $z$). Validation **walk-forward** (train = 2 premiers tiers du temps, test = dernier tiers).


In [ ]:
# --- cible F = d_z(rho0 <u'w'>) et champs grande echelle, restreints 0-15 km ---
mask_z15 = (alt >= 0) & (alt <= Z_TOP_KM * 1000)
alt15 = alt[mask_z15]; zkm15 = alt15/1000.0
n_z15 = alt15.size

# derivees calculees sur la colonne complete PUIS tronquees
F_full   = np.gradient(flux, alt, axis=0)          # (nz, n_stat)
F15      = F_full[mask_z15]
ubar15   = ubar[mask_z15]
dudz15   = dudz[mask_z15]
d2udz215 = d2udz2[mask_z15]

# --- critere de fort cisaillement (sur profil moyen temporel) ---
dudz_absmean = np.abs(dudz15).mean(axis=1)
seuil_shear  = np.percentile(dudz_absmean, 75)     # quantile 75%
strong_z     = dudz_absmean > seuil_shear
lab, nlab = ndi_label(strong_z)
print(f'seuil |d_z ubar| = {seuil_shear:.2e} s^-1  ->  {strong_z.sum()}/{n_z15} niveaux forts')
print(f'{nlab} region(s) CONTINUE(S) en z :')
regions = []
for r in range(1, nlab+1):
    idx = np.where(lab == r)[0]
    regions.append(idx)
    print(f'  region {r}: z = {zkm15[idx[0]]:.2f} - {zkm15[idx[-1]]:.2f} km  ({idx.size} niveaux)')

# --- carte (z,t) des points de fort cisaillement instantane ---
strong_zt = np.abs(dudz15) > seuil_shear
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
mx = np.abs(dudz15).max() + 1e-30
im0 = axes[0].pcolormesh(np.arange(n_stat)+t_stat, zkm15, dudz15,
                         cmap='RdBu_r', shading='auto',
                         norm=TwoSlopeNorm(0, -mx, mx))
axes[0].set_title(r'$\partial_z\bar u(z,t)$'); axes[0].set_ylabel('z (km)')
fig.colorbar(im0, ax=axes[0], pad=0.02)
axes[1].pcolormesh(np.arange(n_stat)+t_stat, zkm15, strong_zt.astype(float),
                   cmap='Greys', shading='auto')
axes[1].set_title(r'zones $|\partial_z\bar u|>$ seuil')
for ax in axes: ax.set_xlabel('t (indice global)'); ax.set_ylim(0, Z_TOP_KM)
fig.suptitle('Localisation des zones de fort cisaillement (0-15 km)')
fig.tight_layout(); plt.show()

In [ ]:
# --- moyennes regionales : profils moyens sur chaque region continue ---
print('Moyennes regionales (moyenne temporelle) :')
for r, idx in enumerate(regions, 1):
    print(f'  region {r} (z={zkm15[idx[0]]:.1f}-{zkm15[idx[-1]]:.1f} km) : '
          f'<F>={F15[idx].mean():+.2e}, <dudz>={dudz15[idx].mean():+.2e}, '
          f'<u>={ubar15[idx].mean():+.2f} m/s')

if regions:
    ridx = int(np.argmax([len(r) for r in regions]))
    r0 = regions[ridx]
    fig, ax = plt.subplots(figsize=(6,4))
    ax.plot(np.arange(n_stat)+t_stat, F15[r0].mean(axis=0), label=r'$\langle F\rangle_{region}$')
    ax.plot(np.arange(n_stat)+t_stat, dudz15[r0].mean(axis=0), label=r'$\langle\partial_z\bar u\rangle$')
    ax.set_xlabel('t'); ax.set_title(f'Moyenne regionale vs temps (region {ridx+1})')
    ax.legend(); fig.tight_layout(); plt.show()

In [ ]:
# ============================================================
# Fermeture locale  F = alpha*ubar + beta*dudz + gamma*d2udz2
# 3 lectures des coefficients, validation walk-forward
# ============================================================
def ridge_fit(X, y, lam=1e-2):
    return np.linalg.solve(X.T @ X + lam*np.eye(X.shape[1]), X.T @ y)

zsel = strong_z.copy()
if zsel.sum() < 3:
    zsel = np.ones(n_z15, dtype=bool)

t_tr = slice(0, 2*n_stat//3); t_te = slice(2*n_stat//3, n_stat)

# ---- (a) coefficients CONSTANTS : fit global sur (z in zsel, t in train) ----
def stack(zmask, tsl):
    X = np.stack([ubar15[zmask][:, tsl].ravel(),
                  dudz15[zmask][:, tsl].ravel(),
                  d2udz215[zmask][:, tsl].ravel()], axis=1)
    y = F15[zmask][:, tsl].ravel()
    return X, y
Xtr, ytr = stack(zsel, t_tr); Xte, yte = stack(zsel, t_te)
mu, sd = Xtr.mean(0), Xtr.std(0)+1e-12
ca = ridge_fit((Xtr-mu)/sd, ytr)
pred_te_a = ((Xte-mu)/sd) @ ca
nse_a = nse(yte, pred_te_a)
coef_a = ca / sd
print('(a) coeffs CONSTANTS (alpha,beta,gamma) =', np.array2string(coef_a, precision=3))
print(f'    NSE test (walk-forward, zones fortes) = {nse_a:+.3f}')

# ---- (b) coefficients par ALTITUDE : un fit par z sur le temps ----
nse_b = np.full(n_z15, np.nan)
for iz in np.where(zsel)[0]:
    Xi = np.stack([ubar15[iz], dudz15[iz], d2udz215[iz]], axis=1)
    m, s = Xi[t_tr].mean(0), Xi[t_tr].std(0)+1e-12
    ci = ridge_fit((Xi[t_tr]-m)/s, F15[iz, t_tr])
    pr = ((Xi[t_te]-m)/s) @ ci
    nse_b[iz] = nse(F15[iz, t_te], pr)
print(f'(b) fit par ALTITUDE : NSE test median (zones fortes) = '
      f'{np.nanmedian(nse_b[zsel]):+.3f}')

# ---- (c) coefficients par INSTANT : un fit par t sur z ----
nse_c = np.full(n_stat, np.nan)
for it in range(n_stat):
    Xi = np.stack([ubar15[zsel, it], dudz15[zsel, it], d2udz215[zsel, it]], axis=1)
    m, s = Xi.mean(0), Xi.std(0)+1e-12
    ci = ridge_fit((Xi-m)/s, F15[zsel, it])
    nse_c[it] = nse(F15[zsel, it], ((Xi-m)/s) @ ci)   # in-sample (peu de ddl)
print(f'(c) fit par INSTANT  : NSE median (in-sample, zones fortes) = '
      f'{np.nanmedian(nse_c):+.3f}')

In [ ]:
# --- visualisation : profils F vrai vs fermeture (a), et NSE(z), NSE(t) ---
Xall = np.stack([ubar15.ravel(), dudz15.ravel(), d2udz215.ravel()], axis=1)
pred_all = (((Xall-mu)/sd) @ ca).reshape(n_z15, n_stat)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
axes[0].plot(F15.mean(1), zkm15, 'k', lw=2, label='F vrai (moy. t)')
axes[0].plot(pred_all.mean(1), zkm15, '--', label='fermeture (a)')
axes[0].fill_betweenx(zkm15, F15.mean(1), where=strong_z, alpha=0.15, color='red',
                      label='zones fortes')
axes[0].axvline(0, color='k', lw=0.5); axes[0].set_ylim(0, Z_TOP_KM)
axes[0].set_xlabel(r"$\partial_z(\rho_0\langle u'w'\rangle)$"); axes[0].set_ylabel('z (km)')
axes[0].legend(fontsize=8); axes[0].set_title('Profil : vrai vs fermeture constante')

axes[1].plot(nse_b, zkm15, 'o-', ms=3)
axes[1].axvline(0, color='k', lw=0.5); axes[1].set_ylim(0, Z_TOP_KM)
axes[1].set_xlabel('NSE test'); axes[1].set_ylabel('z (km)')
axes[1].set_title('(b) NSE par altitude')

axes[2].plot(np.arange(n_stat)+t_stat, nse_c, '.-', ms=4)
axes[2].axhline(0, color='k', lw=0.5)
axes[2].set_xlabel('t'); axes[2].set_ylabel('NSE')
axes[2].set_title('(c) NSE par instant')
fig.tight_layout(); plt.show()

print('\n=== Bilan fermeture locale (zones de fort cisaillement, 0-15 km) ===')
print(f'  (a) coeffs constants   : NSE test  = {nse_a:+.3f}')
print(f'  (b) coeffs par altitude: NSE test  = {np.nanmedian(nse_b[zsel]):+.3f} (median)')
print(f'  (c) coeffs par instant : NSE in-s. = {np.nanmedian(nse_c):+.3f} (median)')

## 6. Validation sur données synthétiques

Avant application aux vraies données, on vérifie sur un cas contrôlé que (i) la **POD corrigée** reconstruit un flux imposé et (ii) la **fermeture locale** retrouve des coefficients connus. Bloc auto-suffisant (numpy seul), exécutable sans les fichiers NetCDF.


In [ ]:
# --- validation synthetique auto-suffisante (numpy deja importe) ---
def _nse(a, b): return 1 - np.sum((a-b)**2)/np.sum((a-a.mean())**2)
rng = np.random.default_rng(0)

# (i) POD des anomalies -> reconstruction d'un flux impose
nzc, nxc, ntc = 30, 48, 40
zc = np.linspace(0, 15000, nzc); xc = np.linspace(0, 48000, nxc)
Zc, Xc = np.meshgrid(zc, xc, indexing='ij')
rho0c = 1.2*np.exp(-zc/8000)
UP = np.zeros((ntc, nzc, nxc)); WP = np.zeros((ntc, nzc, nxc))
for t in range(ntc):
    up = np.zeros((nzc, nxc)); wp = np.zeros((nzc, nxc))
    for k, (zk, amp, ph) in enumerate([(4000,3,0.), (8000,2,1.3), (11000,1.5,2.1)]):
        env = np.exp(-((Zc-zk)/2500)**2); kx = (k+1)*2*np.pi/xc[-1]
        up += amp*env*np.cos(kx*Xc + 0.15*t + ph)
        wp += amp*env*np.sin(kx*Xc + 0.15*t + ph)*1.1
    up += 0.3*rng.standard_normal((nzc, nxc)); wp += 0.3*rng.standard_normal((nzc, nxc))
    UP[t] = up - up.mean(1, keepdims=True); WP[t] = wp - wp.mean(1, keepdims=True)
flux_t = (rho0c[None, :]*(UP*WP).mean(2)).mean(0)
Sy = np.concatenate([UP.reshape(ntc, -1), WP.reshape(ntc, -1)], 1)
Ay = Sy - Sy.mean(0); Cy = Ay@Ay.T/ntc
ev, evc = np.linalg.eigh(Cy); ordy = ev.argsort()[::-1]; evc = evc[:, ordy]
my = evc.T@Ay; my = my/(np.linalg.norm(my, axis=1)+1e-30)[:, None]
ay = Ay@my.T; hlf = nzc*nxc
def _reco(N):
    rec = Sy.mean(0)[None] + ay[:, :N]@my[:N]
    u = rec[:, :hlf].reshape(ntc, nzc, nxc); w = rec[:, hlf:].reshape(ntc, nzc, nxc)
    u -= u.mean(2, keepdims=True); w -= w.mean(2, keepdims=True)
    return (rho0c[None, :]*(u*w).mean(2)).mean(0)
print('(i)  POD anomalies : NSE flux (8 modes) =',
      f'{_nse(flux_t, _reco(8)):+.3f}  (attendu ~0.98)')

# (ii) fermeture locale : la MECANIQUE des moindres carres recupere les coeffs
#      quand les regresseurs sont bien conditionnes (regresseurs independants).
nzz, ntt = 40, 120
g1 = rng.standard_normal((nzz, ntt))    # ~ ubar
g2 = rng.standard_normal((nzz, ntt))    # ~ dudz
g3 = rng.standard_normal((nzz, ntt))    # ~ d2udz2
at, bt, ct = 0.7, -1.3, 0.4
Ff = at*g1 + bt*g2 + ct*g3 + 0.02*rng.standard_normal((nzz, ntt))
X = np.stack([g1.ravel(), g2.ravel(), g3.ravel()], 1); y = Ff.ravel()
c, *_ = np.linalg.lstsq(X, y, rcond=None)
print('(ii) fermeture (regresseurs bien conditionnes) : coeffs recuperes =',
      np.array2string(c, precision=3), ' | vrais = [0.7 -1.3 0.4]')
print('     NSE =', f'{_nse(y, X@c):+.3f}  (attendu ~1.0)   cond(X) =',
      f'{np.linalg.cond(X):.1f}')
print('\nNB physique : sur les vraies donnees, ubar / dudz / d2udz2 sont fortement')
print('collineaires (cond >> 1). La fermeture lineaire locale est donc mal posee ;')
print('c est ce qui limite le NSE de la section 5, pas un bug de code.')
print('\nValidation synthetique OK.')

## 7. Bilan

- **POD corrigée** : la POD sur les anomalies $[u';w']$ (au lieu de $\psi$ moyenné-$y$) **retrouve le flux de Reynolds** (NSE section 3) — le point clé des consignes.
- **$\psi'$ comme relief** : la DCT 2D compresse fortement le relief (section 4) et permet une reconstruction alternative du flux ; comparaison des trois voies (POD $u'w'$, $\psi$ complet, $\psi$ tronqué DCT).
- **Cisaillement & fermeture** : zones $|\partial_z\bar u|>$ seuil repérées, régions continues identifiées, moyennes régionales, fermeture linéaire locale évaluée en (a) constante, (b) par altitude, (c) par instant avec validation walk-forward. Les NSE disent honnêtement dans quelle mesure une fermeture linéaire *grande échelle* explique la divergence de flux (le résidu sous-maille $T_2$ en limite la portée).
- **Validation synthétique** (section 6) : confirme que les deux méthodes fonctionnent quand le signal recherché est présent.
